In [1]:
import pandas as pd 
import os 
import sys 
import glob 
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F
from pyspark.sql import types as T 

In [2]:
spk = SparkSession.builder \
    .appName("data creating ")\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/01 04:58:53 WARN Utils: Your hostname, 2640L, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/01 04:58:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/01 04:58:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [16]:
filepath = "/mnt/d/all_data/20260901/read_data.csv"
spk_df = spk.read.csv(filepath, header=True, inferSchema=True)
spk_df.printSchema()

root
 |-- student_name: string (nullable = true)
 |-- aadhaar_no: long (nullable = true)
 |-- mobile_no: long (nullable = true)
 |-- scheme_amount: integer (nullable = true)
 |-- application_id: string (nullable = true)



In [ ]:
# benificiary_id	installment_id	benificiary_reference_id	benificiary_name	benificiary_mobile_no	adhaar_no	amount	payment_mode_id	district_code	taluk_code	jurisdiction_layer_code	jurisdiction_code	benificiary_account_no	benificiary_bank_name	benificiary_ifsc_code	benificiary_iin_no


In [17]:
col_selct = ["aadhaar_no",
             "application_id",
             "student_name",
             "mobile_no",
             "scheme_amount",
             ]


spk_df = spk_df.select(*col_selct)

In [18]:
spk_df = (
    spk_df
    .withColumn("benificiary_id", F.col("aadhaar_no").cast("string"))
    .withColumn("installment_id", F.lit(1).cast("integer"))
    .withColumn("benificiary_reference_id", F.col("application_id").cast("string"))
    .withColumn("benificiary_name", F.col("student_name").cast("string"))
    .withColumn("benificiary_mobile_no", F.col("mobile_no").cast("string"))
    .withColumn("adhaar_no", F.col("aadhaar_no").cast("string"))
    .withColumn("amount", F.col("scheme_amount").cast("decimal(18,2)"))
    .withColumn("payment_mode_id", F.lit(1).cast("integer"))
    .withColumn("district_code", F.lit(1000).cast("integer"))
    .withColumn("taluk_code", F.lit(1000).cast("integer"))
    .withColumn("jurisdiction_layer_code",F.lit(None))
    .withColumn("jurisdiction_code", F.lit(None))
    .withColumn("benificiary_account_no", F.lit(None))
    .withColumn("benificiary_bank_name", F.lit(None))
    .withColumn("benificiary_ifsc_code", F.lit(None))
    .withColumn("benificiary_iin_no", F.lit(None))
)

In [19]:
select_columns = [
    "benificiary_id",
    "installment_id",
    "benificiary_reference_id",
    "benificiary_name",
    "benificiary_mobile_no",
    "adhaar_no",
    "amount",
    "payment_mode_id",
    "district_code",
    "taluk_code",
    "jurisdiction_layer_code",
    "jurisdiction_code",
    "benificiary_account_no",
    "benificiary_bank_name",
    "benificiary_ifsc_code",
    "benificiary_iin_no"
]

spk_df = spk_df.select(*select_columns)

In [20]:
spk_df = (
    spk_df
    .withColumn("jurisdiction_layer_code", F.lit(None).cast("string"))
    .withColumn("jurisdiction_code", F.lit(None).cast("string"))
    .withColumn("benificiary_account_no", F.lit(None).cast("string"))
    .withColumn("benificiary_bank_name", F.lit(None).cast("string"))
    .withColumn("benificiary_ifsc_code", F.lit(None).cast("string"))
    .withColumn("benificiary_iin_no", F.lit(None).cast("string"))
    .select(*select_columns)
)

In [21]:
spk_df = spk_df.select(
    *[F.col(c).cast("string").alias(c) for c in select_columns]
)

In [22]:
spk_df.printSchema()

root
 |-- benificiary_id: string (nullable = true)
 |-- installment_id: string (nullable = false)
 |-- benificiary_reference_id: string (nullable = true)
 |-- benificiary_name: string (nullable = true)
 |-- benificiary_mobile_no: string (nullable = true)
 |-- adhaar_no: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- payment_mode_id: string (nullable = false)
 |-- district_code: string (nullable = false)
 |-- taluk_code: string (nullable = false)
 |-- jurisdiction_layer_code: string (nullable = true)
 |-- jurisdiction_code: string (nullable = true)
 |-- benificiary_account_no: string (nullable = true)
 |-- benificiary_bank_name: string (nullable = true)
 |-- benificiary_ifsc_code: string (nullable = true)
 |-- benificiary_iin_no: string (nullable = true)



In [23]:
pandas_df = spk_df.toPandas()

/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [24]:
pandas_df = pandas_df.astype("string")

In [25]:
csv_path = "/mnt/d/all_data/20260901/mbc_gis_beneficiary_data.csv"

pandas_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8"
)

print(f"CSV file created: {csv_path}")

CSV file created: /mnt/d/all_data/20260901/mbc_gis_beneficiary_data.csv
